<a href="https://colab.research.google.com/github/jma199/telco-customer-churn/blob/main/Prepare_Dashboard_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparing Files for Dashboard

The preferred way to structure a data source for dashboarding with a software such as Power BI is star schema. Three of the five data tables will be merged as a customer table. There will also be a services table and a population table.

In [1]:
import pandas as pd

# Load files
services = pd.read_excel("Telco_customer_churn_services.xlsx")
population = pd.read_excel("Telco_customer_churn_population.xlsx")

demographics = pd.read_excel("Telco_customer_churn_demographics.xlsx")
location = pd.read_excel("Telco_customer_churn_location.xlsx")
status = pd.read_excel("Telco_customer_churn_status.xlsx")

In [2]:
dim_customers = demographics.merge(status, on="Customer ID", how="left")
dim_customers = dim_customers.merge(location, on="Customer ID", how="left")

In [3]:
def remove_merged_duplicate_columns(df):
    """
    Find and drop both '_x' and '_y' versions of duplicated columns
    after a merge operation.

    Args:
        df (pd.DataFrame): The DataFrame to process.

    Returns:
        pd.DataFrame: The DataFrame with both '_x' and '_y' versions of duplicated columns dropped.
    """

    # Find duplicated columns (contain "_")
    duplicate_base_columns = set()
    for col in df.columns:
        if col.endswith('_x'):
            base_name = col[:-2]
            if f'{base_name}_y' in df.columns:
                duplicate_base_columns.add(base_name)

    list_duplicated_base_columns = list(duplicate_base_columns)

    if not list_duplicated_base_columns:
        print("No duplicated columns found to remove.")
        return df

    print(f"Identified duplicated columns: {list_duplicated_base_columns}")

    # Logic from drop_duplicate_merged_columns
    columns_to_drop = []
    for base_col in list_duplicated_base_columns:
        if f'{base_col}_x' in df.columns:
            columns_to_drop.append(f'{base_col}_x')
        if f'{base_col}_y' in df.columns:
            columns_to_drop.append(f'{base_col}_y')

    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"Dropped columns: {columns_to_drop}")
    else:
        print("No columns identified to drop.")

    return df


print(f"\nInitial columns before combined removal ({len(dim_customers.columns)}):")
print(dim_customers.columns.tolist())

df_cleaned = remove_merged_duplicate_columns(dim_customers)

print(f"\nFinal columns after combined removal ({len(df_cleaned.columns)}):")
print(df_cleaned.columns.tolist())


Initial columns before combined removal (27):
['Customer ID', 'Count_x', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Count_y', 'Quarter', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Category', 'Churn Reason', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']
Identified duplicated columns: ['Count']
Dropped columns: ['Count_x', 'Count_y']

Final columns after combined removal (25):
['Customer ID', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Quarter', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Category', 'Churn Reason', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']


In [4]:
# Export to use for dashboarding
dim_customers.to_csv("dim_customers.csv", index=False)
services.to_csv("fact_services.csv", index=False)
population.to_csv("dim_population.csv", index=False)